In [13]:
import matplotlib.pyplot as plt
import numpy as np
import scipy as sp
import scipy.interpolate
from pylab import rcParams
%matplotlib inline
from matplotlib.cm import ScalarMappable
import os
import re

In [1]:
import os
import re


outcar_file = 'OUTCAR.phon'
output_folder = 'phonon_displacements' # Folder to store the output files

# --- Setup Output Directory ---
if not os.path.exists(output_folder):
    os.makedirs(output_folder)
    print(f"Created output directory: {output_folder}")

# --- Regular Expressions for Parsing OUTCAR ---
# Matches lines like "1 f = ..." or "250 f/i= ..." to find new phonon modes.
mode_start_regex = re.compile(r'^\s*(\d+)\s+f(?:/i)?\s*=\s')
# Matches the header line "X Y Z dx dy dz" to signal the start of data.
data_header_regex = re.compile(r'^\s*X\s+Y\s+Z')
# Extracts all floating-point numbers from a line (handles various formats and spacing).
float_value_regex = re.compile(r'[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?')

# --- Main Extraction Logic ---
current_output_file = None
is_collecting_data = False
line_counter = 0 # Initialize line counter for each output file

try:
    with open(outcar_file, 'r') as infile:
        print(f"Processing OUTCAR: {outcar_file}")
        for line in infile:
            # Check if a new phonon mode block begins
            mode_match = mode_start_regex.match(line)
            if mode_match:
                # Close the previous file if open
                if current_output_file:
                    current_output_file.close()
                
                # Get the mode number and open a new file for it
                mode_number = mode_match.group(1)
                filepath = os.path.join(output_folder, f'{mode_number}.txt')
                current_output_file = open(filepath, 'w')
                is_collecting_data = False # Reset data collection flag
                line_counter = 0 # Reset line counter for the new file
                #print(f"Opened file for mode {mode_number} to extract displacements.")
                continue # Move to the next line in OUTCAR

            # Check for the data header line
            if data_header_regex.match(line):
                is_collecting_data = True # Start collecting data from next line
                continue # Move to the next line in OUTCAR

            # If collecting data for the current mode
            if is_collecting_data and current_output_file:
                # Stop collecting if an empty line is encountered (end of data block)
                if not line.strip():
                    is_collecting_data = False
                    continue # Move to the next line in OUTCAR

                # Extract all numbers from the line
                numbers_found = float_value_regex.findall(line)
                
                # Determine where the actual X, Y, Z, dx, dy, dz values start.
                # OUTCAR often includes an atom index before the coordinates.
                start_index = 0
                # If the first number looks like an integer (atom index)
                # AND there are more than 6 numbers in total (indicating an extra index column)
                if numbers_found and numbers_found[0].isdigit() and len(numbers_found) > 6:
                    start_index = 1 # Skip the first number (the atom index)

                # Ensure we have at least 6 numerical values (X, Y, Z, dx, dy, dz)
                # We are interested in the last three (dx, dy, dz)
                if len(numbers_found) - start_index >= 6:
                    # Extract ONLY the last three relevant numerical strings (dx, dy, dz)
                    # These are at indices start_index + 3, start_index + 4, start_index + 5
                    displacement_data = numbers_found[start_index + 3 : start_index + 6]
                    
                    # Increment line counter and write to file with line number
                    line_counter += 1
                    current_output_file.write(f"{line_counter}\t{' '.join(displacement_data)}\n")
                else:
                    print(f"Warning: Line in current mode did not contain enough data for displacements: '{line.strip()}'")

finally:
    # Ensure the last opened file is closed when done or if an error occurs
    if current_output_file:
        current_output_file.close()

print(f"\nPhonon mode displacement (dx dy dz) files with line numbers extracted to '{output_folder}/'.")


Processing OUTCAR: OUTCAR.phon

Phonon mode displacement (dx dy dz) files with line numbers extracted to 'phonon_displacements/'.
